# Model Evaluation and Error Analysis

This notebook evaluates the performance of the KNN classifier on the `clean_waveform_dataset` and analyzes common failure modes.

In [1]:
import sys
from pathlib import Path
import json
import random
from collections import Counter

# Add parent directory to path to import inference_demo
sys.path.append(str(Path("..")))

from clean_waveform_benchmark.inference_demo import KNearestNeighbors, load_rows

## 1. Load Data and Train Models

In [2]:
data_path = Path("../clean_waveform_benchmark/clean_waveform_dataset.json")
rows = load_rows(data_path)

# Filter for noisy variants as training/test data
eval_rows = [r for r in rows if r["variant_role"] == "noisy_variant"]
random.Random(42).shuffle(eval_rows)

split = int(len(eval_rows) * 0.7)
train_rows, test_rows = eval_rows[:split], eval_rows[split:]

site_model = KNearestNeighbors("site_class", k=5).fit(train_rows)
fill_model = KNearestNeighbors("fill_state", k=5).fit(train_rows)

## 2. Overall Performance

In [3]:
site_score = site_model.score(test_rows)
fill_score = fill_model.score(test_rows)

print(f"Site Class Accuracy: {site_score['accuracy']:.3f}")
print(f"Fill State Accuracy: {fill_score['accuracy']:.3f}")

Site Class Accuracy: 1.000
Fill State Accuracy: 0.941


## 3. Confusion Matrix Analysis

Let's look at where the `fill_state` model is making mistakes.

In [4]:
print("Confusion (Actual, Predicted):")
for (actual, pred), count in fill_score['confusion'].items():
    if actual != pred:
        print(f"Mistake: {actual} -> {pred}: {count}")
    else:
        print(f"Correct: {actual} -> {pred}: {count}")

Confusion (Actual, Predicted):
('partial', 'partial'): 18
('empty', 'empty'): 14
('full', 'full'): 16
('full', 'partial'): 3


## 4. Error Deep Dive

Why did the model misclassify these records?

In [5]:
mistakes = [r for r in test_rows if fill_model.predict_one(r)[0] != r['labels']['fill_state']]

print("Mistake Detail:")
for r in mistakes[:3]:
    pred, conf, neighbors = fill_model.predict_one(r)
    print(f"Record ID: {r['record_id']}")
    print(f"Actual: {r['labels']['fill_state']}, Predicted: {pred} (Conf: {conf:.2f})")
    print(f"Pattern Notes: {r['pattern_notes']}")
    print("---")

Mistake Detail:
Record ID: retail_full__v002
Actual: full, Predicted: partial
Pattern Notes: Earlier resistance, slightly higher energy, noisy baseline.
---
